# Final-Run Walkthrough — Evolutionary Factor Researcher

This notebook documents **every stage of one evolutionary factor-research run** end-to-end, showing
the concrete inputs and outputs of each step: the data panel, the knowledge retrieval, the exact
prompts sent to the LLM and its replies, the debate transcript, the generated factor code, the
4-axis fitness evaluation with all diagnostics, the archive state per generation, the
progressive-reveal / rescore events, the final curation + deflation arithmetic, the persisted book,
and the LLM token/cost report.

It is **read-only over a completed run's artifacts** — point `RUN_DIR` at any prerun produced by
`run_factor_evolution.py` (see `docs/research-evolution/FINAL_RUN_PLAN.md`). Execute it first
against the small smoke run (verification gate D3) and later against a real ladder arm.

Design references: `docs/research-evolution/DESIGN.md`, `FINAL_RUN_PLAN.md`.


In [ ]:
from pathlib import Path
import json, os
import pandas as pd

# ---- Parameters -------------------------------------------------------------
CONFIG_FILE = 'quant.config.nasdaq100_2010.yaml'   # data config the run used
RUN_DIR = Path('data/workspaces/fmp_archive_equity_nasdaq100pit/preruns/SMOKE')  # <- point at a prerun
EVO = RUN_DIR / 'evolution'
assert EVO.exists(), f'no evolution dir under {RUN_DIR} — run a smoke run first (plan §D3)'

def jload(p):
    return json.loads(Path(p).read_text()) if Path(p).exists() else None
def jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()] if p.exists() else []


## 1. Run configuration & provenance

`run_config.json` is the full `EvolutionRunConfig` (search shape, operator mix, progressive-reveal
schedule knobs, curation/deflation settings). `manifest.json` carries the headline provenance:
which LLM (per role), which engine, how many trials were billed, and the total token/cost usage.


In [ ]:
run_cfg = jload(EVO / 'run_config.json')
manifest = jload(RUN_DIR / 'manifest.json')
display(pd.Series(manifest).to_frame('manifest'))
pd.Series({k: v for k, v in (run_cfg or {}).items() if not isinstance(v, (dict, list))}).to_frame('run_config')


## 2. The data panel (what the researcher is allowed to see)

The panel is loaded through the pluggable data layer with the **point-in-time membership mask**
(cells are NaN when the ticker was not an index member on that date) and the config's end date
enforces the **two-year forward reserve** — nothing after it exists in-process. The DATA CONTEXT
section of every prompt is generated from exactly these fields.


In [ ]:
os.environ['QF_CONFIG_FILE'] = CONFIG_FILE
from quant_fund_agent.config import get_settings
from quant_fund_agent.data.panel import load_panel
from quant_fund_agent.data import usable_fields
settings = get_settings()
fields = sorted(usable_fields(settings))
print(f'{len(fields)} usable fields:', fields[:20], '...')
panel = load_panel(settings.data, fields=['close'])
close = panel['close']
print(f'panel: {close.shape[0]} bars x {close.shape[1]} tickers, density {close.notna().mean().mean():.1%}')
print('range:', close.index[0].date(), '->', close.index[-1].date(), ' (must end >= 2y before today)')
close.notna().sum(axis=1).plot(title='PIT cross-section size over time', figsize=(9,2.5));


### 2b. The progressive-reveal schedule

With `--progressive-reveal`, the dev window is revealed block-by-block: **expanding IS + sliding
VAL**, and a final `test_frac` tail that is *never* revealed during the run (touch-once TEST).
Part of each reveal was never queried by earlier selection — prevention of the generational
ratchet, not detection. `progressive.json` records the final frontier.


In [ ]:
prog = jload(EVO / 'progressive.json')
prog


## 3. Knowledge grounding: retrieval → mechanism groups

GraphRAG arms resolve mechanism groups from the knowledge graph's under-covered communities;
each group carries a **focus brief** spliced into that group's seeding and mutation prompts.
Papers are retrieved under the run's cutoff date and `data_scope` mask (fundamentals papers are
invisible to OHLCV-only runs).


In [ ]:
state = jload(EVO / 'state.json') or {}
groups = run_cfg.get('mechanism_groups') if run_cfg else None
print('retrieval mode:', run_cfg.get('retrieval') if run_cfg else '?')
print('mechanism groups:', run_cfg.get('n_mechanism_groups') if run_cfg else '?')
# group focus briefs are stamped into the loop's group context; inspect state/lineage for them
[e for e in jsonl(EVO / 'lineage.jsonl') if e.get('event') == 'group_focus'][:5]


## 4. The LLM calls: exact prompts in, exact replies out

With `QF_LLM_TRANSCRIPT=1` the run records every LLM call (role, prompt, response, tokens) to
`evolution/llm_transcript.jsonl`. Below: one **seed brainstorm** call, one **hypothesis** call, one
**skeptic/moderator debate** exchange and one **codegen** call, printed in full.


In [ ]:
tx = jsonl(EVO / 'llm_transcript.jsonl')
print(f'{len(tx)} recorded LLM calls; roles:', pd.Series([t['role'] for t in tx]).value_counts().to_dict() if tx else '{}')
def show_call(role, i=0):
    calls = [t for t in tx if t['role'] == role]
    if not calls: return print(f'(no {role} calls recorded)')
    c = calls[i]
    print('='*100); print(f'ROLE={role}  tokens_in={c.get("input_tokens")} tokens_out={c.get("output_tokens")}')
    print('-'*40, 'PROMPT', '-'*40); print(c['prompt'][:6000])
    print('-'*40, 'REPLY', '-'*41); print(c['response'][:4000])
show_call('brainstorm')


In [ ]:
show_call('hypothesis'); show_call('debate'); show_call('codegen')


## 5. From idea to factor: codegen, validation, in-memory compile

Every accepted idea becomes a `BaseFactor` module (validated: allow-listed imports, declared
`inputs` within the data scope, its own `prediction_horizon`). Candidates compile in-memory —
zero file/registry churn during the search; only the final book is materialised.


In [ ]:
factor_db = jload(RUN_DIR / 'factors' / 'factor_db.json')
recs = (factor_db or {}).get('factors', factor_db or [])
print(f'{len(recs)} persisted factors')
one = recs[0] if isinstance(recs, list) else list(recs.values())[0]
print(json.dumps({k: one.get(k) for k in ('id','description','inputs','prediction_horizon')}, indent=2)[:1200])
print((one.get('code') or '')[:2000])


## 6. Fitness: the 4-axis Pareto vector + gates, in full

`ObjectiveVector.AXES = (marginal_value, independence, parsimony, structural_novelty)`:
- **marginal_value** — LOCO marginal ΔOOS-IC vs the current book (nonlinear combiner), minus the
  window-jitter plateau penalty and perturbation probe, ± hypothesis sign bonus (all IC-scale).
- **independence** — residual (orthogonalised) IC on IS∪VAL.
- **parsimony** — −AST complexity.
- **structural_novelty** — canonical-AST weighted-subtree distance to the nearest book member.
Gates: coverage (+ optional cost). Deflation is a *publish* filter, not a search gate.
Below: one candidate's complete objective + gates + diagnostics as recorded in lineage.


In [ ]:
lineage = jsonl(EVO / 'lineage.jsonl')
evals = [e for e in lineage if e.get('objective')]
print(f'{len(evals)} evaluated candidates in lineage')
json.dumps(evals[0], indent=2)[:3000] if evals else '(empty)'


## 7. Selection & the archive, generation by generation

Parents: binary tournament on (front rank, crowding) within a deme; migration on a ring strictly
inside each mechanism group. The archive is the per-group **front-1** set, now with a crowding-
culled cap (`archive_cap_per_group`); evictions are first-class lineage events. `gen_quality.jsonl`
tracks the book's quality per generation — the **evolutionary effect** curve.


In [ ]:
gq = pd.DataFrame(jsonl(EVO / 'gen_quality.jsonl'))
display(gq.tail())
if not gq.empty:
    ax = gq.set_index('generation')[['mean_marginal_value','max_marginal_value']].plot(figsize=(9,3), title='Archive quality across generations')
    gq.set_index('generation')['archive_size_total'].plot(secondary_y=True, ax=ax, style='k--');


In [ ]:
evicts = pd.DataFrame([e for e in lineage if e.get('event') == 'archive_evict'])
evicts.groupby('reason').size() if not evicts.empty else 'no evictions'


## 8. Progressive reveal: prequential OOS + rescore drift

At each reveal the archive is scored on the **about-to-be-revealed block** (honest OOS — no
selection ever saw it): `prequential.jsonl`. Then the window advances and every member is
re-scored (`rescore` lineage rows, with before/after objectives). This separates the
**data-reveal effect** (drift across reveals) from the **evolutionary effect** (across generations).


In [ ]:
preq = pd.DataFrame(jsonl(EVO / 'prequential.jsonl'))
display(preq)
if not preq.empty and 'combined_oos_ic' in preq:
    preq.dropna(subset=['combined_oos_ic']).plot(x='reveal_index', y='combined_oos_ic', marker='o', figsize=(9,3), title='Honest prequential OOS IC of the book, per reveal');


In [ ]:
res = [e for e in lineage if e.get('event') == 'rescore']
drift = pd.DataFrame([{'generation': e['generation'], 'genome': e['genome_id'][:8],
                        'marg_before': (e.get('objective_before') or {}).get('marginal_value'),
                        'marg_after': (e.get('objective_after') or {}).get('marginal_value')} for e in res])
display(drift.head(10))
fails = [e for e in lineage if e.get('event') == 'rescore_failed']
print(f'{len(res)} rescores, {len(fails)} failed rescores')


## 9. End-of-run: curation → publish deflation → the final book

The kept_pool (every gate-passer, ever) is curated by greedy forward-selection on combined VAL IC;
the selected book's combined statistic is then **deflated for N_trials** (every scored candidate
billed) and pruned by LOCO marginal contribution until it passes. Failures now fail *closed*.
Each persisted factor's metadata records the full decision trail.


In [ ]:
meta = one.get('metadata', {})
json.dumps(meta.get('evolution', meta), indent=2)[:2500]


## 10. Cost report

Per-role token and dollar accounting from the usage meter (also in the manifest). This is the
per-LLM cost table for the thesis comparison.


In [ ]:
usage = (manifest or {}).get('llm_usage') or (jload(EVO / 'run_config.json') or {}).get('llm_usage')
pd.DataFrame(usage).T if usage else '(no usage recorded)'


## 11. Verification checklist for this run

- [ ] panel ends ≥ 2 years before today (forward reserve intact)
- [ ] membership mask density in the expected band (~0.45 for Nasdaq-100 PIT)
- [ ] progressive reveal ON; prequential rows have `reveal_index` stamps; no duplicate generations
- [ ] archive never exceeded `archive_cap_per_group`; evictions logged
- [ ] every persisted factor: gates passed, deflation recorded, provenance stamped (model + roles)
- [ ] cost meter total consistent with provider dashboard
